# meta-openenv — T4 Colab training (SFT + GRPO)

1. **Runtime → Change runtime type → T4 GPU** (or better).
2. The setup cell clones `VanshGupta18/corporate-compliance-env` by default; change `GITHUB_USER` only if you are using a fork.
3. Run cells in order.

No WebSocket server is required; rollouts use in-process `ComplianceEnv`.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

GITHUB_USER = "VanshGupta18"  # change if you are using a fork
REPO = "corporate-compliance-env"
BRANCH = "main"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO}.git"
WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()
REPO_DIR = WORKDIR / REPO

if GITHUB_USER == "YOUR_GITHUB_USER":
    raise ValueError("Set GITHUB_USER to your GitHub username before running.")

WORKDIR.mkdir(parents=True, exist_ok=True)
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
elif REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 1) Install training dependencies

The `cuda-bindings` pip warning on Colab is harmless. This cell also removes optional `torchcodec`, `torchao`, and `sentence-transformers` packages because they are unnecessary for text-only QLoRA and can break Unsloth imports on Colab. **After this cell finishes, use Runtime → Restart session**, then run the verify cell below.

In [ ]:
import sys

print("Python:", sys.version)

# OpenEnv runtime deps (no HF training pins)
%pip install -q -r requirements.txt

# Unsloth + TRL (install only — import in the NEXT cell after runtime restart)
%pip install -q "unsloth==2026.4.6"
%pip install -q "trl==0.24.0" "datasets>=3.4.1,<4.4.0"

# Text-only training does not need these optional packages. On Colab they can
# break Unsloth imports via sentence-transformers / Transformers optional paths.
%pip uninstall -y -q torchcodec torchao sentence-transformers

print("Install complete.")
print("Next: Runtime → Restart session, then run the verify cell below.")

In [ ]:
import importlib
import sys

import torch

print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Switch Colab runtime to T4 GPU before training.")
print("GPU:", torch.cuda.get_device_name(0))

for module in ("torch", "trl", "transformers", "peft", "accelerate", "datasets"):
    pkg = importlib.import_module(module)
    print(f"{module}:", getattr(pkg, "__version__", "ok"))

# Import Unsloth last; this is the real compatibility check.
import unsloth
print("unsloth:", getattr(unsloth, "__version__", "ok"))

from training.training_utils import grpo_supports_rollout_func

print("TRL rollout_func:", grpo_supports_rollout_func())
print("Stack ready — continue with dataset prep / dry runs.")

## 2) Optional: regenerate claims if splits are missing

In [ ]:
import pathlib
import subprocess
import sys

splits = list(pathlib.Path("data/splits").glob("*.json"))
if len(splits) < 3:
    subprocess.run(
        [
            sys.executable,
            "data/generate_dataset.py",
            "--train-per-diff",
            "120",
            "--val-per-diff",
            "40",
            "--test-per-diff",
            "40",
            "--seed",
            "42",
        ],
        check=True,
    )
else:
    print("data/splits present, skipping generate_dataset")

## 3) Prepare SFT data + dry runs

In [ ]:
import subprocess
import sys

commands = [
    [sys.executable, "training/prepare_data.py", "--episodes-per-task", "40", "--split", "train"],
    [sys.executable, "training/sft_train.py", "--dry-run"],
    [sys.executable, "training/grpo_train.py", "--dry-run", "--curriculum-stage", "stage_1_easy"],
    [sys.executable, "training/smoke_test.py"],
]
for command in commands:
    print("$", " ".join(command))
    subprocess.run(command, check=True)

### Reminder

If you skipped the post-install runtime restart, do **Runtime → Restart session** now, then rerun setup + verify cells before training.

In [ ]:
# Install order matters on Colab: requirements.txt → unsloth → trl (see cell above).

## 4) SFT warm start (Unsloth QLoRA, T4 defaults)

In [ ]:
!python training/sft_train.py \
  --model-id unsloth/Qwen2.5-3B-Instruct-bnb-4bit \
  --dataset-path training/data/sft_dataset.jsonl \
  --output-dir training/checkpoints/sft \
  --max-length 512 \
  --batch-size 1 \
  --grad-accum 8

## 5) GRPO curriculum (stages 1 → 2 → 3)

If OOM: lower `--num-generations` to `1` or reduce `--max-train-steps`.

In [ ]:
!python training/grpo_train.py \
  --sft-checkpoint training/checkpoints/sft \
  --curriculum-stage stage_1_easy \
  --output-dir training/checkpoints/grpo_stage1 \
  --max-seq-length 512 \
  --num-generations 2 \
  --batch-size 1 \
  --grad-accum 8 \
  --max-train-steps 100

In [ ]:
!python training/grpo_train.py \
  --sft-checkpoint training/checkpoints/grpo_stage1 \
  --curriculum-stage stage_2_medium \
  --output-dir training/checkpoints/grpo_stage2 \
  --max-seq-length 512 \
  --num-generations 2 \
  --batch-size 1 \
  --grad-accum 8 \
  --max-train-steps 100

In [ ]:
!python training/grpo_train.py \
  --sft-checkpoint training/checkpoints/grpo_stage2 \
  --curriculum-stage stage_3_hard \
  --output-dir training/checkpoints/grpo \
  --max-seq-length 512 \
  --num-generations 2 \
  --batch-size 1 \
  --grad-accum 8 \
  --max-train-steps 200

## 6) Evaluate checkpoint (in-process env)

In [ ]:
!python training/eval_checkpoint.py \
  --checkpoint training/checkpoints/grpo \
  --split validation \
  --episodes 10 \
  --episode-log-file training/logs/episodes.jsonl \
  --clear-log

## 7) Optional: publish adapter to Hugging Face

Create a model repo on the Hub, then set `HF_REPO_ID` and run the next cell.

In [ ]:
HF_REPO_ID = "YOUR_HF_USERNAME/compliance-grpo-adapter"  # edit

if HF_REPO_ID.startswith("YOUR_"):
    print("Set HF_REPO_ID to publish; skipping.")
else:
    import subprocess

    from huggingface_hub import notebook_login

    notebook_login()
    subprocess.run(
        [
            "python",
            "training/publish_adapter.py",
            "--checkpoint",
            "training/checkpoints/grpo",
            "--repo-id",
            HF_REPO_ID,
        ],
        check=True,
    )